<a href="https://colab.research.google.com/github/ritikarpawar/Tensorflow/blob/main/pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install the tensorflow-model-optimization library if not already installed
!pip install -q tensorflow-model-optimization

import tensorflow as tf
import tensorflow_model_optimization as tfmot
import numpy as np

# Load data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train/255.0, x_test/255.0
x_train = x_train[..., None]
x_test = x_test[..., None]

# Baseline model
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=2, verbose=0)
base_acc = model.evaluate(x_test, y_test, verbose=0)[1]
print("Baseline Accuracy:", base_acc)

# Pruning
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

# Define the pruning parameters
epochs = 1
initial_sparsity = 0.5
final_sparsity = 0.9
begin_step = 0
end_step = np.ceil(x_train.shape[0] / 32).astype(np.int32) * epochs

pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=initial_sparsity,
        final_sparsity=final_sparsity,
        begin_step=begin_step,
        end_step=end_step
    )
}

# Apply pruning to the model
pruned_model = prune_low_magnitude(model, **pruning_params)

pruned_model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

# Add the UpdatePruningStep callback
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_model.fit(x_train, y_train, epochs=epochs, verbose=0, callbacks=callbacks)
pruned_acc = pruned_model.evaluate(x_test, y_test, verbose=0)[1]
print("Pruned Accuracy:", pruned_acc)

# Quantization
converter = tf.lite.TFLiteConverter.from_keras_model(pruned_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

print("Quantized Model Size (KB):", len(tflite_model)/1024)

Baseline Accuracy: 0.9785000085830688
Pruned Accuracy: 0.968500018119812
Quantized Model Size (KB): 850.8046875
